# Coqui XTTS-v2 Voice Cloning in Google Colab

This notebook allows you to use Coqui XTTS-v2 for text-to-speech with voice cloning using a reference audio file.

## 1. Install Dependencies

## 2. Imports and Device Setup

In [1]:
from TTS.api import TTS
import torch
import os

# --- User Configuration for Device ---
# Set your desired device: "cuda" for GPU (if available), or "cpu".
TARGET_DEVICE = "cuda"  # Options: "cuda" or "cpu"
# -------------------------------------

# Determine the device to use
print(f"Target device specified: {TARGET_DEVICE}")
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if TARGET_DEVICE == "cuda" and cuda_available:
    device = "cuda"
elif TARGET_DEVICE == "cuda" and not cuda_available:
    device = "cpu"
    print("CUDA was targeted but is not available. Falling back to CPU.")
else:
    device = "cpu"

print(f"Using device: {device}")

Target device specified: cuda
CUDA available: False
CUDA was targeted but is not available. Falling back to CPU.
Using device: cpu


## 3. User Inputs

### Upload Reference Audio
Upload your short reference audio file (e.g., a 5-15 second WAV file). Make sure the filename does not contain spaces or special characters for simplicity.

### Specify Text to Synthesize and Language

In [2]:
# Edit this line to change the text you want to synthesize.
text_to_speak = "Hello, this is a test of your custom voice workflow in Google Colab."

# Set the language for XTTS (e.g., "en", "es", "fr", "de", etc.)
language_to_use = "en"

output_wav_path = "xtts_colab_generated_speech.wav"

print(f"Text to synthesize: {text_to_speak}")
print(f"Language: {language_to_use}")
print(f"Output file will be: {output_wav_path}")

Text to synthesize: Hello, this is a test of your custom voice workflow in Google Colab.
Language: en
Output file will be: xtts_colab_generated_speech.wav


## 4. Initialize TTS Model and Perform Inference

### Multilingual XTTS-v2 with speaker embedding

In [1]:
POEM = """Do not go gentle into that good night,
Old age should burn and rave at close of day;
Rage, rage against the dying of the light.

Though wise men at their end know dark is right,
Because their words had forked no lightning they
Do not go gentle into that good night.

Good men, the last wave by, crying how bright
Their frail deeds might have danced in a green bay,
Rage, rage against the dying of the light.

Wild men who caught and sang the sun in flight,
And learn, too late, they grieved it on its way,
Do not go gentle into that good night.

Grave men, near death, who see with blinding sight
Blind eyes could blaze like meteors and be gay,   
Rage, rage against the dying of the light.

And you, my father, there on the sad height,
Curse, bless, me now with your fierce tears, I pray.
Do not go gentle into that good night.
Rage, rage against the dying of the light.
"""
def split_text_by_n_sentences(text, n):
    sentences = []
    i = 0
    concat_sentences = ""
    for sentence in text.split("\n"):
        concat_sentences += sentence + " "
        if i % n == 0 and i != 0:
            sentences.append(concat_sentences)
            concat_sentences = ""
        i += 1
    return sentences

sentences_to_speak = split_text_by_n_sentences(POEM, 3)

print(sentences_to_speak)

['Do not go gentle into that good night, Old age should burn and rave at close of day; Rage, rage against the dying of the light.  ', 'Though wise men at their end know dark is right, Because their words had forked no lightning they Do not go gentle into that good night. ', ' Good men, the last wave by, crying how bright Their frail deeds might have danced in a green bay, ', 'Rage, rage against the dying of the light.  Wild men who caught and sang the sun in flight, ', 'And learn, too late, they grieved it on its way, Do not go gentle into that good night.  ', 'Grave men, near death, who see with blinding sight Blind eyes could blaze like meteors and be gay,    Rage, rage against the dying of the light. ', ' And you, my father, there on the sad height, Curse, bless, me now with your fierce tears, I pray. ', 'Do not go gentle into that good night. Rage, rage against the dying of the light.  ']


In [28]:
import os
import torch
import torchaudio
from TTS.api import TTS

# ── User params ─────────────────────────────────────────────
REFERENCE_WAV = "long_cumberbatch.wav"
OUTPUT_WAV    = "cumberbatch_xtts_embedded.wav"

LANGUAGE = "en"
# ─────────────────────────────────────────────────────────────

device = "cpu"

# 1️⃣ Load the high-level TTS wrapper
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
# grab the underlying model+config
model  = tts.synthesizer.tts_model.to(device)
config = tts.synthesizer.tts_model.config

# 2️⃣ Zero-shot: compute GPT-conditioning + speaker latent from your WAV
print("Computing conditioning latents…")
gpt_latent, speaker_embed = model.get_conditioning_latents(
    audio_path=[REFERENCE_WAV]
)

# 3️⃣ Inference
print("Synthesizing full poem…")
wav_tensors = []
for sentence in sentences_to_speak:
    out = model.inference(
        sentence,
        LANGUAGE,
        gpt_latent,
        speaker_embed,
        temperature=0.7,
        speed=1.0,
    )

    # 4️⃣ Save
    wav_tensor = torch.tensor(out["wav"]).cpu()
    wav_tensors.append(wav_tensor)

final_wav = torch.cat(wav_tensors, dim=0).unsqueeze(0)
sr = config.audio["sample_rate"]
torchaudio.save(OUTPUT_WAV, final_wav, sr)
print(f"✅ Saved to {OUTPUT_WAV}")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
Computing conditioning latents…
Synthesizing full poem…
✅ Saved to cumberbatch_xtts_embedded.wav
